# 05 - Fixed word replacement (A) and best-of-3 selection (C), v8

**Two ideas, tested together on top of your last run (same Drive folder `quick2`):**
* **A. Fix SIRA's word replacement.** The new attack families `sira2` and `random2` stop the doubled words ("lengths lengths"), the stray brackets ("(1D) (1D)") and the wrong "a/an", and never mask two neighbouring words.
* **C. Best of 3.** For each text we make 3 versions of the best attack so far (paraphrase + a few random word changes) and keep the one that beats the detectors and reads best.

**How we keep the result honest.** The version is picked using the original detectors and judge A. It is then graded by (1) two *held-out* detectors that took no part in the choice, and (2) a second, *independent* quality judge (a different model family, different wording). The final success rate uses those.

**Everything is set up. You paste and edit nothing.** Before you start: **Runtime -> Change runtime type -> T4 GPU**, and switch on notebook access for `HF_TOKEN` (key icon, left sidebar). Run the cells from top to bottom, one at a time. Every stage saves its work; you can stop a stage with the square stop button and run it again later, it continues where it stopped.

## 1. Log in

In [ ]:
from google.colab import userdata
from huggingface_hub import login, whoami
login(token=userdata.get("HF_TOKEN"), add_to_git_credential=False)
print("HF user:", whoami()["name"])

## 2. Get the code (always a fresh copy)
If Colab asks to restart the session, do it, then rerun cells 1 and 2.

In [ ]:
!rm -rf /content/sira-lab
!git clone -q https://github.com/deymonp/sira-lab.git /content/sira-lab
%cd /content/sira-lab
!pip install -q -r requirements.txt
!pip install -q -e . --no-deps

## 3. Settings
Colab will ask you to allow access to Google Drive. Allow it. Check that the last line says `True`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

RUN = "/content/drive/MyDrive/sira_runs/quick2"     # same folder as before: texts, attacks and scores are re-used

import os
from sira_lab.config import load_config
from sira_lab.lab.run import run_generate, run_attack, run_evaluate
from sira_lab.lab.judge import run_judge, run_judge_b
from sira_lab.lab.select import select_best
cfg = load_config("configs/lab_v8.yaml")
print("prepared first:", cfg.lab.prep_attacks)
print("attacks:", cfg.lab.attacks)
print("texts from the last run found:", os.path.exists(f"{RUN}/generated.jsonl"))

## 4. Stage 2: the new attacks
Two extra paraphrases are made first (inputs for the other versions), then the chain versions, `sira2` and `random2`. Texts already done are skipped. The cell prints its running time at the end. Please keep that number.

In [ ]:
import time
t0 = time.time()
run_attack(cfg, RUN)
print(f"Stage 2 took {(time.time() - t0) / 60:.1f} minutes")

## 5. Stage 3: score everything with all detectors
Scores from earlier runs are re-used, so only the new texts are scored. Progress is saved after each detector.

In [ ]:
import time, pandas as pd
from IPython.display import Image, display
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30); pd.set_option("display.max_rows", 300)
t0 = time.time()
df = run_evaluate(cfg, RUN)
print(f"Stage 3 took {(time.time() - t0) / 60:.1f} minutes")
display(df[["detector", "algo", "attack", "detect_rate", "changed_mean", "sim_median"]].round(3))

## 6. Stage 4: quality judge A
Scores the new texts for fluency and meaning (1 to 5). Earlier texts are skipped.

In [ ]:
import time
t0 = time.time()
run_judge(cfg, RUN)
print(f"Stage 4 took {(time.time() - t0) / 60:.1f} minutes")

## 7. Stage 5: pick the best of 3 for every text
This needs no GPU. For each text it keeps the version that the original detectors miss and judge A rates best. Held-out detectors are **not** used here.

In [ ]:
BASE = "paraphrase+random2@0.15"
sel = select_best(cfg, RUN, base=BASE, n=3)

## 8. Stage 6: the independent quality judge B
A different model family with different wording grades the picked texts, so the judge that chose the winner is not the judge that grades it. It downloads one more model (about 8 GB). If it cannot be used, the notebook says so and falls back to a weaker second opinion.

In [ ]:
import time
t0 = time.time()
KINDS = ["best3", "paraphrase+random2@0.15", "paraphrase+random@0.15", "paraphrase", "sira@0.3", "sira2@0.3", "random2@0.3"]
label = run_judge_b(cfg, RUN, kinds=KINDS)
print(f"Stage 6 took {(time.time() - t0) / 60:.1f} minutes")

## 9. The results
`success_B` is the final grade (independent judge B). `success_A` uses judge A, which also helped pick the winners, so it is optimistic for `best3`. `evade_zs_ho` is the held-out detectors.

In [ ]:
df = run_evaluate(cfg, RUN)       # re-uses all stored scores; now includes best3
from sira_lab.lab.analysis import success_table, plot_success
stA = success_table(RUN)
stB = success_table(RUN, quality_file="quality_b.csv")
stA.to_csv(f"{RUN}/success_A.csv", index=False)
stB.to_csv(f"{RUN}/success_B.csv", index=False)
plot_success(stB, f"{RUN}/success.png")
cols = ["algo", "attack", "n", "evade_wm", "evade_zs", "evade_zs_ho", "evade_all", "quality_ok", "success", "success_lo", "success_hi", "fluency_mean", "meaning_mean"]
print("judge B model:", open(f"{RUN}/quality_b_model.txt").read(), "\n")
print("FINAL (judge B):")
print(stB[cols].sort_values(["algo", "success"], ascending=[True, False]).round(3).to_string(index=False))
print("\nWith judge A (optimistic for best3):")
print(stA[cols].sort_values(["algo", "success"], ascending=[True, False]).round(3).to_string(index=False))
display(Image(f"{RUN}/success.png"))

## 10. Do the two judges agree?
A quick sanity check on the texts both judges scored. A correlation near 0 would mean the quality scores are mostly noise.

In [ ]:
qa = pd.read_csv(f"{RUN}/quality.csv"); qb = pd.read_csv(f"{RUN}/quality_b.csv")
m = qa.merge(qb, on=["kind", "algo", "id"], suffixes=("_A", "_B")).dropna(subset=["fluency_A", "fluency_B"])
print("texts scored by both:", len(m))
print("correlation fluency A vs B: %.2f" % m["fluency_A"].corr(m["fluency_B"]))
mm = m.dropna(subset=["meaning_A", "meaning_B"])
print("correlation meaning A vs B: %.2f  (n=%d)" % (mm["meaning_A"].corr(mm["meaning_B"]), len(mm)))
print(m.groupby("kind")[["fluency_A", "fluency_B", "meaning_A", "meaning_B"]].mean().round(2).to_string())

## 11. What to send me
Open the Drive folder `My Drive / sira_runs / quick2` in your browser, download these files and attach them in the chat:

`success_B.csv`, `success_A.csv`, `quality.csv`, `quality_b.csv`, `quality_b_model.txt`, `results.csv`, `scores.csv`, `success.png`

Also tell me the times printed under Stages 2, 3, 4 and 6, and copy the output of cell 10. When you are done, click **Runtime -> Disconnect and delete runtime**.

In [ ]:
!ls -la {RUN}